# HW4: Speaker Prediction with Self-Attention

Task: speaker prediction from preprocessed utterance features.

- Training samples: 56,666
- Testing samples: 4,000
- Metric: categorical accuracy
- Main idea: represent each utterance as a sequence and use self-attention to aggregate speaker-discriminative temporal information.


## Download the Dataset

The KaggleHub cache is redirected to `HW4/data/`. If the dataset already exists locally, the cell reuses it and skips the download.


In [ ]:
# !pip install kagglehub==1.0.1

import inspect
import os
import tarfile
from pathlib import Path
from zipfile import ZipFile

DEFAULT_DOWNLOAD_DIR = Path("machine-learning/hung-yi-lee-machine-learning/HW4")
if not DEFAULT_DOWNLOAD_DIR.exists():
    DEFAULT_DOWNLOAD_DIR = Path.cwd()
DEFAULT_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR.resolve()
DATA_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR / "data"
DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set this to a local dataset directory if the competition files were downloaded manually.
DATA_DOWNLOAD_ROOT = None
# DATA_DOWNLOAD_ROOT = "/path/to/ml2023springhw4"

# Optional: set this if the token is not under the current user's home directory.
KAGGLE_CREDENTIALS_DIR = None
# KAGGLE_CREDENTIALS_DIR = "/path/to/.kaggle"

KAGGLE_CONFIG_DIR = Path(
    KAGGLE_CREDENTIALS_DIR
    or os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle")
).expanduser().resolve()
KAGGLE_ACCESS_TOKEN = KAGGLE_CONFIG_DIR / "access_token"
KAGGLE_JSON = KAGGLE_CONFIG_DIR / "kaggle.json"
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_CONFIG_DIR)
os.environ["KAGGLEHUB_CACHE"] = str(DATA_DOWNLOAD_DIR)


def find_existing_data_dir(root: Path) -> Path | None:
    candidates = [root]
    if root.exists():
        candidates.extend(candidate for candidate in root.glob("**/*") if candidate.is_dir())

    for candidate in candidates:
        if (candidate / "metadata.json").exists() and (candidate / "testdata.json").exists():
            return candidate.resolve()
    return None


def has_kaggle_credentials() -> bool:
    has_env_credentials = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_access_token = KAGGLE_ACCESS_TOKEN.exists()
    has_json_credentials = KAGGLE_JSON.exists()
    if has_access_token:
        KAGGLE_ACCESS_TOKEN.chmod(0o600)
    if has_json_credentials:
        KAGGLE_JSON.chmod(0o600)
    return has_env_credentials or has_access_token or has_json_credentials


def download_competition(handle: str, output_dir: Path) -> str:
    import kagglehub

    signature = inspect.signature(kagglehub.competition_download)
    if "output_dir" in signature.parameters:
        return kagglehub.competition_download(handle, output_dir=str(output_dir))
    return kagglehub.competition_download(handle)


def extract_archives(root: Path) -> None:
    for zip_path in root.glob("**/*.zip"):
        target_dir = zip_path.with_suffix("")
        if not target_dir.exists():
            with ZipFile(zip_path) as archive:
                archive.extractall(target_dir)

    for tar_path in list(root.glob("**/*.tar")) + list(root.glob("**/*.tar.gz")) + list(root.glob("**/*.tgz")):
        suffix = ".tar.gz" if tar_path.name.endswith(".tar.gz") else tar_path.suffix
        target_dir = tar_path.parent / tar_path.name.removesuffix(suffix)
        if not target_dir.exists():
            with tarfile.open(tar_path) as archive:
                archive.extractall(target_dir)


if DATA_DOWNLOAD_ROOT is None:
    existing_data_dir = find_existing_data_dir(DATA_DOWNLOAD_DIR)
    if existing_data_dir is not None:
        download_path = existing_data_dir
        path = str(download_path)
    else:
        if not has_kaggle_credentials():
            raise FileNotFoundError(
                f"Kaggle credentials were not found. Put the KaggleHub token at {KAGGLE_ACCESS_TOKEN}, "
                f"put legacy API credentials at {KAGGLE_JSON}, or set KAGGLE_USERNAME and KAGGLE_KEY "
                "before starting this notebook. Do not paste the token into this notebook."
            )

        try:
            downloaded_root = Path(download_competition("ml2023springhw4", DATA_DOWNLOAD_DIR)).expanduser().resolve()
        except Exception as error:
            error_name = error.__class__.__name__
            error_text = str(error)
            if error_name == "UnauthenticatedError":
                raise RuntimeError(
                    "KaggleHub still cannot authenticate. Make sure the current Jupyter kernel can see "
                    f"{KAGGLE_ACCESS_TOKEN} or {KAGGLE_JSON}, or restart the kernel after setting environment credentials. "
                    "Also join the competition and accept its rules on Kaggle before re-running this cell."
                ) from error
            if "403" in error_text and "rules" in error_text.lower():
                raise PermissionError(
                    "Kaggle authentication succeeded, but this account does not have permission to download "
                    "the competition data yet. Open https://www.kaggle.com/competitions/ml2023springhw4/rules, "
                    "join the competition, accept the rules, then re-run this cell."
                ) from error
            raise

        extract_archives(downloaded_root)
        download_path = find_existing_data_dir(downloaded_root) or find_existing_data_dir(DATA_DOWNLOAD_DIR)
        if download_path is None:
            raise FileNotFoundError("Could not find metadata.json and testdata.json after download.")
        path = str(download_path)
else:
    download_path = Path(DATA_DOWNLOAD_ROOT).expanduser().resolve()
    path = str(download_path)

print("Data directory:", download_path)
print("KaggleHub cache:", os.environ["KAGGLEHUB_CACHE"])


## Setup and Model Overview

For an utterance feature sequence

$$
X=[x_1,\dots,x_T]\in\mathbb{R}^{T\times d},
$$

self-attention computes contextual frames

$$
H=\operatorname{TransformerEncoder}(XW+p),
$$

then attentive statistics pooling gives

$$
\alpha_t=\operatorname{softmax}(w^T\tanh(W_a h_t)),\qquad
\mu=\sum_t\alpha_t h_t,
$$

$$
\sigma=\sqrt{\sum_t\alpha_t(h_t-\mu)^2+\epsilon},\qquad
z=[\mu;\sigma].
$$

The classifier predicts the speaker from $z$.


In [ ]:
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import wandb
except ImportError:
    wandb = None


@dataclass
class Config:
    seed: int = 42
    segment_len: int = 128
    max_eval_segments: int = 8
    batch_size: int = 256
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 50
    early_stopping_patience: int = 8
    label_smoothing: float = 0.1
    gradient_clip_norm: float = 5.0
    valid_ratio: float = 0.1
    d_model: int = 256
    nhead: int = 4
    num_layers: int = 4
    dim_feedforward: int = 1024
    dropout: float = 0.1
    num_workers: int = 4
    prefetch_factor: int = 2
    gpu_id: int = 0
    use_amp: bool = True
    show_batch_progress: bool = False
    model_filename: str = "hw4_speaker_transformer_best.pt"
    submission_filename: str = "submission_hw4.csv"
    use_wandb: bool = False
    wandb_project: str = "hung-yi-lee-machine-learning"
    wandb_run_name: str = "hw4-speaker-self-attention"


CONFIG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_id = CONFIG.gpu_id if 0 <= CONFIG.gpu_id < gpu_count else 0
        if gpu_id != CONFIG.gpu_id:
            print(f"Requested CUDA GPU {CONFIG.gpu_id}, but only {gpu_count} GPU(s) are available. Falling back to cuda:0.")
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")
        torch.backends.cudnn.benchmark = True
        device = torch.device(f"cuda:{gpu_id}")
        torch.cuda.set_device(device)
        print(f"Using CUDA GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
        return device

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("Using Apple Metal Performance Shaders (MPS).")
        return torch.device("mps")

    print("Using CPU. GPU acceleration is not available in this environment.")
    return torch.device("cpu")


def to_device(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.to(device, non_blocking=(device.type == "cuda"))


def dataloader_kwargs(shuffle: bool = False) -> dict:
    kwargs = {
        "batch_size": CONFIG.batch_size,
        "shuffle": shuffle,
        "num_workers": CONFIG.num_workers,
        "pin_memory": device.type == "cuda",
    }
    if CONFIG.num_workers > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = CONFIG.prefetch_factor
    return kwargs


def init_wandb():
    if not CONFIG.use_wandb:
        return None
    if wandb is None:
        raise ImportError("Install wandb or set CONFIG.use_wandb=False.")
    return wandb.init(
        project=CONFIG.wandb_project,
        name=CONFIG.wandb_run_name,
        config=asdict(CONFIG),
    )


set_seed(CONFIG.seed)
device = choose_device()
wandb_run = init_wandb()

print(f"Using device: {device}")
print(asdict(CONFIG))


## Locate and Inspect the Dataset

The notebook expects the common HW4 JSON layout: `metadata.json`, `mapping.json`, and `testdata.json`, with utterance features stored as `.pt` tensors.


In [ ]:
def find_data_dir() -> Path:
    roots: list[Path] = [Path.cwd(), Path.cwd().parent, Path(".").resolve()]
    if "path" in globals():
        roots.append(Path(path))

    candidates: list[Path] = []
    for root in roots:
        candidates.append(root)
        if root.exists():
            candidates.extend(candidate for candidate in root.glob("**/*") if candidate.is_dir())

    for candidate in candidates:
        if (candidate / "metadata.json").exists() and (candidate / "testdata.json").exists():
            return candidate.resolve()

    raise FileNotFoundError("Could not find metadata.json and testdata.json. Run the Kaggle download cell first.")


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


DATA_DIR = find_data_dir()
metadata = load_json(DATA_DIR / "metadata.json")
testdata = load_json(DATA_DIR / "testdata.json")
mapping = load_json(DATA_DIR / "mapping.json") if (DATA_DIR / "mapping.json").exists() else {}

speakers = metadata.get("speakers", {})
if not speakers:
    raise ValueError("metadata.json is expected to contain a non-empty 'speakers' object.")

speaker_to_index = mapping.get("speaker2id") or {speaker: index for index, speaker in enumerate(sorted(speakers))}
index_to_speaker = {int(index): speaker for speaker, index in speaker_to_index.items()}
num_speakers = len(speaker_to_index)


def normalize_feature_path(feature_path: str | Path) -> Path:
    feature_path = Path(feature_path)
    if feature_path.is_absolute():
        return feature_path
    return DATA_DIR / feature_path


def utterance_feature_path(item) -> str:
    if isinstance(item, str):
        return item
    if isinstance(item, dict):
        return item.get("feature_path") or item.get("path") or item.get("file")
    raise TypeError(f"Unsupported utterance item: {type(item)}")


train_items: list[tuple[Path, int]] = []
for speaker, utterances in speakers.items():
    speaker_index = int(speaker_to_index[speaker])
    for item in utterances:
        train_items.append((normalize_feature_path(utterance_feature_path(item)), speaker_index))

raw_test_items = testdata.get("utterances") or testdata.get("testdata") or testdata
if isinstance(raw_test_items, dict):
    raw_test_items = list(raw_test_items.values())
test_paths = [normalize_feature_path(utterance_feature_path(item)) for item in raw_test_items]

print("Data directory:", DATA_DIR)
print("Training utterances:", len(train_items))
print("Testing utterances:", len(test_paths))
print("Speakers:", num_speakers)
print("Example feature:", train_items[0][0])


## Prepare Speaker Datasets

Training uses random fixed-length segments. Validation and inference use deterministic segments to reduce variance.


In [ ]:
def load_feature(feature_path: Path) -> torch.Tensor:
    feature = torch.load(feature_path, map_location="cpu")
    if isinstance(feature, dict):
        for key in ("mel", "feature", "feat"):
            if key in feature:
                feature = feature[key]
                break
        else:
            raise KeyError(f"Could not find a feature tensor key in {feature_path}")
    feature = torch.as_tensor(feature, dtype=torch.float32)

    if feature.ndim != 2:
        raise ValueError(f"Expected a 2-D feature tensor, got shape {tuple(feature.shape)} from {feature_path}")
    if feature.shape[0] < feature.shape[1] and feature.shape[0] <= 128:
        feature = feature.transpose(0, 1)

    mean = feature.mean(dim=0, keepdim=True)
    std = feature.std(dim=0, keepdim=True).clamp_min(1e-5)
    return (feature - mean) / std


def crop_or_pad(feature: torch.Tensor, segment_len: int, random_crop: bool) -> torch.Tensor:
    frames = feature.size(0)
    if frames >= segment_len:
        if random_crop:
            start = random.randint(0, frames - segment_len)
        else:
            start = max((frames - segment_len) // 2, 0)
        return feature[start:start + segment_len]

    pad = feature.new_zeros(segment_len - frames, feature.size(1))
    return torch.cat([feature, pad], dim=0)


def make_eval_segments(feature: torch.Tensor, segment_len: int, max_segments: int) -> torch.Tensor:
    frames = feature.size(0)
    if frames <= segment_len:
        return crop_or_pad(feature, segment_len, random_crop=False).unsqueeze(0)

    segment_count = min(max_segments, math.ceil(frames / segment_len))
    starts = np.linspace(0, frames - segment_len, num=segment_count, dtype=np.int64)
    return torch.stack([feature[start:start + segment_len] for start in starts.tolist()])


def split_train_valid(items: list[tuple[Path, int]], valid_ratio: float) -> tuple[list[tuple[Path, int]], list[tuple[Path, int]]]:
    rng = random.Random(CONFIG.seed)
    by_speaker: dict[int, list[Path]] = {}
    for feature_path, speaker_index in items:
        by_speaker.setdefault(speaker_index, []).append(feature_path)

    train_split: list[tuple[Path, int]] = []
    valid_split: list[tuple[Path, int]] = []
    for speaker_index, paths in by_speaker.items():
        paths = paths[:]
        rng.shuffle(paths)
        valid_size = max(1, int(len(paths) * valid_ratio)) if len(paths) > 1 else 0
        valid_split.extend((path, speaker_index) for path in paths[:valid_size])
        train_split.extend((path, speaker_index) for path in paths[valid_size:])

    rng.shuffle(train_split)
    rng.shuffle(valid_split)
    return train_split, valid_split


class SpeakerDataset(Dataset):
    def __init__(self, items: list[tuple[Path, int]], training: bool) -> None:
        self.items = items
        self.training = training

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        feature_path, speaker_index = self.items[index]
        feature = load_feature(feature_path)
        segment = crop_or_pad(feature, CONFIG.segment_len, random_crop=self.training)
        return segment, torch.tensor(speaker_index, dtype=torch.long)


train_split, valid_split = split_train_valid(train_items, CONFIG.valid_ratio)
input_dim = load_feature(train_split[0][0]).size(1)

train_dataset = SpeakerDataset(train_split, training=True)
valid_dataset = SpeakerDataset(valid_split, training=False)
train_loader = DataLoader(train_dataset, **dataloader_kwargs(shuffle=True))
valid_loader = DataLoader(valid_dataset, **dataloader_kwargs())

print("Train utterances:", len(train_split))
print("Valid utterances:", len(valid_split))
print("Input dimension:", input_dim)


## Build and Train the Self-Attention Classifier

Self-attention learns frame-to-frame dependencies inside an utterance; attentive statistics pooling converts variable acoustic evidence into a speaker embedding.


In [ ]:
class AttentiveStatsPool(nn.Module):
    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.attention(x).squeeze(-1), dim=1).unsqueeze(-1)
        mean = torch.sum(weights * x, dim=1)
        variance = torch.sum(weights * (x - mean.unsqueeze(1)).pow(2), dim=1).clamp_min(1e-6)
        std = torch.sqrt(variance)
        return torch.cat([mean, std], dim=1)


class SpeakerTransformer(nn.Module):
    def __init__(self, input_dim: int, num_speakers: int) -> None:
        super().__init__()
        self.prenet = nn.Sequential(
            nn.Linear(input_dim, CONFIG.d_model),
            nn.LayerNorm(CONFIG.d_model),
            nn.GELU(),
            nn.Dropout(CONFIG.dropout),
        )
        self.position = nn.Parameter(torch.zeros(1, CONFIG.segment_len, CONFIG.d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=CONFIG.d_model,
            nhead=CONFIG.nhead,
            dim_feedforward=CONFIG.dim_feedforward,
            dropout=CONFIG.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=CONFIG.num_layers)
        self.pool = AttentiveStatsPool(CONFIG.d_model)
        self.classifier = nn.Sequential(
            nn.LayerNorm(CONFIG.d_model * 2),
            nn.Linear(CONFIG.d_model * 2, CONFIG.d_model),
            nn.GELU(),
            nn.Dropout(CONFIG.dropout),
            nn.Linear(CONFIG.d_model, num_speakers),
        )
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.prenet(x) + self.position[:, :x.size(1)]
        h = self.encoder(h)
        z = self.pool(h)
        return self.classifier(z)


model = SpeakerTransformer(input_dim=input_dim, num_speakers=num_speakers).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG.max_epochs, eta_min=CONFIG.learning_rate * 0.01)
amp_enabled = CONFIG.use_amp and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")
print(f"AMP enabled: {amp_enabled}")
if wandb_run is not None:
    wandb.watch(model, log="gradients", log_freq=100)
model


In [ ]:
def autocast_context():
    if amp_enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def run_epoch(model: nn.Module, loader: DataLoader, training: bool, desc: str) -> tuple[float, float]:
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    progress = tqdm(loader, desc=desc, leave=False, disable=not CONFIG.show_batch_progress)

    for x_batch, y_batch in progress:
        x_batch = to_device(x_batch)
        y_batch = to_device(y_batch)

        with torch.set_grad_enabled(training):
            with autocast_context():
                logits = model(x_batch)
                loss = criterion(logits, y_batch)

        if training:
            optimizer.zero_grad(set_to_none=True)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG.gradient_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG.gradient_clip_norm)
                optimizer.step()

        total_loss += loss.item() * len(x_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(x_batch)

    return total_loss / total, correct / total


history: list[dict[str, float]] = []
best_valid_accuracy = 0.0
epochs_without_improvement = 0
checkpoint_path = Path(CONFIG.model_filename)
epoch_bar = tqdm(range(1, CONFIG.max_epochs + 1), desc="Training", dynamic_ncols=True)

for epoch in epoch_bar:
    train_loss, train_accuracy = run_epoch(model, train_loader, training=True, desc=f"train {epoch:03d}")
    valid_loss, valid_accuracy = run_epoch(model, valid_loader, training=False, desc=f"valid {epoch:03d}")
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    is_best = valid_accuracy > best_valid_accuracy
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
        "learning_rate": current_lr,
        "is_best": is_best,
    })

    if wandb_run is not None:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/accuracy": train_accuracy,
            "valid/loss": valid_loss,
            "valid/accuracy": valid_accuracy,
            "learning_rate": current_lr,
        })

    if is_best:
        best_valid_accuracy = valid_accuracy
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": asdict(CONFIG),
            "speaker_to_index": speaker_to_index,
            "valid_accuracy": valid_accuracy,
            "epoch": epoch,
        }, checkpoint_path)
    else:
        epochs_without_improvement += 1

    epoch_bar.set_postfix({
        "train_acc": f"{train_accuracy:.4f}",
        "valid_acc": f"{valid_accuracy:.4f}",
        "best": f"{best_valid_accuracy:.4f}",
        "lr": f"{current_lr:.2e}",
        "stale": epochs_without_improvement,
    })

    if epochs_without_improvement >= CONFIG.early_stopping_patience:
        break

epoch_bar.close()
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Best validation accuracy: {checkpoint['valid_accuracy']:.4f} at epoch {checkpoint['epoch']}")
if wandb_run is not None:
    wandb.summary["best_valid_accuracy"] = checkpoint["valid_accuracy"]
    wandb.finish()


In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["valid_loss"], label="valid")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["valid_accuracy"], label="valid")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()


## Predict the Test Set and Export the Submission

For each utterance, several deterministic segments are evaluated and their logits are averaged.


In [ ]:
@torch.no_grad()
def predict_one(feature_path: Path) -> int:
    feature = load_feature(feature_path)
    segments = make_eval_segments(feature, CONFIG.segment_len, CONFIG.max_eval_segments).to(device)
    logits_sum = torch.zeros(num_speakers, device=device)

    for start in range(0, len(segments), CONFIG.batch_size):
        batch = segments[start:start + CONFIG.batch_size]
        with autocast_context():
            logits = model(batch)
        logits_sum += logits.float().sum(dim=0)

    return int(logits_sum.argmax().item())


predictions = [predict_one(feature_path) for feature_path in tqdm(test_paths, desc="Predicting")]

sample_submission_paths = sorted(DATA_DIR.glob("**/*submission*.csv"))
if sample_submission_paths:
    submission = pd.read_csv(sample_submission_paths[0])
    label_column = "Category" if "Category" in submission.columns else submission.columns[-1]
    submission[label_column] = predictions
else:
    submission = pd.DataFrame({"Id": np.arange(len(predictions)), "Category": predictions})

submission_path = Path(CONFIG.submission_filename)
submission.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path.resolve()}")
submission.head()
